In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [2]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

In [3]:
from datasets import load_dataset
dataset = load_dataset("databricks/databricks-dolly-15k")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [4]:
import pandas as pd

df = dataset["train"].to_pandas()

df.head()

,instruction,context,response,category
0,When did Virgin Australia start operating?,"Virgin Australia, the trading name of Virgin A...",Virgin Australia commenced services on 31 Augu...,closed_qa
1,Which is a species of fish? Tope or Rope,,Tope,classification
2,Why can camels survive for long without water?,,Camels use the fat in their humps to keep them...,open_qa
3,"Alice's parents have three daughters: Amy, Jes...",,The name of the third daughter is Alice,open_qa
4,When was Tomoaki Komorida born?,Komorida was born in Kumamoto Prefecture on Ju...,"Tomoaki Komorida was born on July 10,1981.",closed_qa


In [5]:
df = df[df["category"].isin(["open_qa", "general_qa", "classification", "closed_qa"])]
df["category"].value_counts()

,count
category,
open_qa,3742
general_qa,2191
classification,2136
closed_qa,1773


In [ ]:
def create_balanced_dataset(df):

    # Count the instances of "closed_qa"
    num_spam = df[df["category"] == "closed_qa"].shape[0]

    # Randomly sample other instances to match the number of "closed_qa" instances
    label1_subset = df[df["category"] == "general_qa"].sample(num_spam, random_state=123)
    label2_subset = df[df["category"] == "classification"].sample(num_spam, random_state=123)

    # Combine other subsets with 0
    balanced_df = pd.concat([label1_subset, label2_subset, df[df["category"] == "closed_qa"]])

    return balanced_df

balanced_df = create_balanced_dataset(df)
print(balanced_df["category"].value_counts())

category
general_qa        1773
classification    1773
closed_qa         1773
Name: count, dtype: int64


In [7]:
data = []

for index, row in balanced_df.iterrows():
    data.append({
        "Instruction": row["instruction"],
        "Input": row["context"],
        "Output": row["response"]
    })

data[50]

{'Instruction': 'What are Jindo dogs like?',
 'Input': '',
 'Output': 'Jindos are extremely loyal, and territorial. Unlike most dogs, Jindos do not play fetch, and rarely bark. They are also very independent, and some would say they march to the beat of their own drum. When meeting strangers they tend to be very reserved, but are highly affectionate to their owners. It is very hard to earn their trust, but once you do Jindos make loving companions.'}

In [8]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(data)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from unsloth import FastModel
import torch


model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B",
    dtype = None, # None for auto detection
    max_seq_length = 1024, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

==((====))==  Unsloth 2026.9.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [11]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",
)

In [12]:
from datasets import Dataset # Add this import

def formatting_prompts_func(examples):
    instructions = examples["Instruction"]
    inputs       = examples["Input"]
    outputs      = examples["Output"]

    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # ۱. اگر بخش ورودی (کانتکست) خالی نبود، آن را با دستور ترکیب کن
        if input_text and input_text.strip():
            user_content = f"{instruction}\n\nInput:\n{input_text}"
        else:
            user_content = instruction

        # ۲. ساختار استاندارد پیام‌ها
        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]

        # ۳. اعمال تمپلیت چت مدل Gemma
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        # حذف توکن تکراری bos در صورت نیاز
        text = text.removeprefix('')
        texts.append(text)

    return {"text": texts}

# Convert the list to a datasets.Dataset object
dataset = Dataset.from_list(dataset)

# اعمال تغییرات روی دیتاست
dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/5319 [00:00<?, ? examples/s]

In [13]:
print(dataset[100]["text"])

<bos><|turn>user
What are the boroughs of New York City?<turn|>
<|turn>model
There are five boroughs in New York City (NYC). The five boroughs are Manhattan, Brooklyn, Queens, the Bronx, and Staten Island.<turn|>



In [14]:
train_portion = int(len(dataset) * 0.85)  # 85% for training
test_portion = int(len(dataset) * 0.1)    # 10% for testing
val_portion = len(dataset) - train_portion - test_portion  # Remaining 5% for validation

train_data =dataset[:train_portion]
test_data = dataset[train_portion:train_portion + test_portion]
val_data =  dataset[train_portion + test_portion:]

In [15]:
from datasets import Dataset

train = Dataset.from_dict(train_data)
val = Dataset.from_dict(val_data)
test = Dataset.from_dict(test_data)

In [16]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # Should leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

In [17]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train,
    eval_dataset = val, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 90,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        eval_strategy = "steps",            # یا evaluation_strategy در نسخه‌های قدیمی‌تر
        eval_steps = 15,
        weight_decay = 0.002,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4521 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/267 [00:00<?, ? examples/s]

In [18]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

Map:   0%|          | 0/4521 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4521 [00:00<?, ? examples/s]

Unsloth: Removed 23 out of 4521 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map:   0%|          | 0/267 [00:00<?, ? examples/s]

Filter:   0%|          | 0/267 [00:00<?, ? examples/s]

Unsloth: Removed 8 out of 267 samples from eval_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


In [19]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
9.18 GB of memory reserved.


In [20]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,498 | Num Epochs = 1 | Total steps = 90
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 12,668,928 of 5,135,846,944 (0.25% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_

Step,Training Loss,Validation Loss
15,1.295430,0.955163
30,0.763782,0.923722
45,1.153942,0.906729
60,1.333548,0.910268
75,1.055690,0.900061
90,1.120947,0.898742


Filter:   0%|          | 0/259 [00:00<?, ? examples/s]

In [21]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1723.0084 seconds used for training.
28.72 minutes used for training.
Peak reserved memory = 11.174 GB.
Peak reserved memory for training = 1.994 GB.
Peak reserved memory % of max memory = 76.729 %.
Peak reserved memory for training % of max memory = 13.692 %.


In [22]:
model.save_pretrained("/content/drive/MyDrive/gemma_4_lora")  # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/gemma_4_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/gemma_4_lora/tokenizer_config.json.


['/content/drive/MyDrive/gemma_4_lora/processor_config.json']

In [24]:
test[0]

{'Instruction': 'According to this paragraph, what was the most common type of castle in England following the Norman Conquest?',
 'Input': 'There are nine castles in Greater Manchester, a metropolitan county in North West England. They consist of four motte-and-bailey castles, three fortified manor houses, an enclosure castle, and a possible shell keep. A motte-and-bailey castle is characterised by two elements: the motte is an artificial mound with a wooden stockade and stronghold on top, usually a stone keep or tower, while the bailey is a defended enclosure adjacent to the motte, typically enclosed by a ditch and a bank topped by a timber palisade or stone wall. Motte-and-bailey castles were the most common type of castle in England following the Norman Conquest. A shell keep was a motte with a stone wall rather than a wooden stockade on top; there would have been no tower within the walls. Four of Greater Manchester\'s castles are scheduled monuments: Buckton, Bury, Radcliffe Towe

In [36]:
test1 = test[:5]

type(test1)

dict

In [37]:
import json
import torch
from tqdm import tqdm
from unsloth import FastLanguageModel

# ۱. بهینه‌سازی مدل برای حالت استنتاج سریع در Unsloth
FastLanguageModel.for_inference(model)

results = []
output_file = "test_predictions.json"

print(f"Starting inference on {len(test)} samples...")
test1 = test[:5]
test1 = Dataset.from_dict(test1)

for sample in tqdm(test1, desc="Evaluating Test Set"):
    instruction = sample["Instruction"]
    input_text  = sample.get("Input", "")
    ground_truth = sample.get("Output", "")

    # ۲. ساخت متن کانتکست کاربر
    if input_text and input_text.strip():
        user_content = f"{instruction}\n\nInput:\n{input_text}"
    else:
        user_content = instruction

    messages = [
        {"role": "user", "content": [{"type": "text", "text": user_content}]}
    ]

    # ۳. اعمال تمپلیت چت جما همراه با تگ شروع مدل (add_generation_prompt=True)
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,  # اضافه کردن تگ model
        return_tensors="pt"
    ).to("cuda")

    prompt_len = inputs.shape[1]

    # ۴. تولید متن خروجی
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            use_cache=True,
            temperature=0.1,         # دمای پایین برای پاسخ‌های دقیق و قطعی
            repetition_penalty=1.15  # جلوگیری از تکرار لوپ‌های کلمات
        )

    # ۵. استخراج خالص پاسخ مدل (حذف پرامپت ورودی)
    generated_tokens = outputs[0][prompt_len:]
    prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # ۶. اضافه کردن به لیست خروجی‌ها
    results.append({
        "instruction": instruction,
        "input": input_text,
        "ground_truth": ground_truth,
        "prediction": prediction
    })

# ۷. ذخیره تمام نتایج در فایل JSON با انکودینگ utf-8
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\nCompleted! Successfully saved predictions to '{output_file}'")

Starting inference on 531 samples...


Evaluating Test Set: 100%|██████████| 5/5 [03:38<00:00, 43.79s/it]


Completed! Successfully saved predictions to 'test_predictions.json'


In [ ]:
# if True:
#     from unsloth import FastModel
#     model, tokenizer = FastModel.from_pretrained(
#         model_name = "gemma_4_lora", # YOUR MODEL YOU USED FOR TRAINING
#         max_seq_length = 2048,
#         load_in_4bit = True,
#     )

# messages = [{
#     "role": "user",
#     "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
# }]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
#     tokenize = True,
#     return_dict = True,
# ).to("cuda")

# from transformers import TextStreamer
# _ = model.generate(
#     **inputs,
#     max_new_tokens = 128, # Increase for longer outputs!
#     # Recommended Gemma-4 settings!
#     temperature = 1.0, top_p = 0.95, top_k = 64,
#     streamer = TextStreamer(tokenizer, skip_prompt = True),
# )